# DEM Analysis with AbovePy

This notebook demonstrates a complete DEM analysis workflow:
searching for tiles, downloading, mosaicking, visualizing elevation,
computing a hillshade, and creating a composite overlay.

We'll work with DEM Phase 3 data near Frankfort, KY — the state capital,
located in the Kentucky River gorge with dramatic terrain.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import abovepy

## Search and Download DEM Tiles

We'll search for DEM Phase 3 tiles covering the Frankfort area and download them.

In [ ]:
# Search for DEM tiles in the Frankfort area
frankfort_bbox = (-84.9, 38.15, -84.8, 38.25)
tiles = abovepy.search(bbox=frankfort_bbox, product="dem_phase3")

print(f"Found {len(tiles)} DEM tiles")
print(tiles[["tile_id", "asset_url"]].head())

# Download tiles
paths = abovepy.download(tiles, output_dir="./dem_data")
print(f"\nDownloaded {len(paths)} files")

## Create Mosaic

Combine the downloaded tiles into a single VRT (Virtual Raster).
VRT is the default format — it's a lightweight XML file that references
the original tiles without duplicating data.

In [ ]:
# Mosaic tiles into a VRT
vrt_path = abovepy.mosaic(paths, output="./dem_data/frankfort_dem.vrt")
print(f"Mosaic created: {vrt_path}")

## Visualize Elevation

Read the mosaic and display the elevation data with a colorbar.
KyFromAbove DEMs are in EPSG:3089 (Kentucky Single Zone, US feet).

In [ ]:
# Read the mosaic
elevation, profile = abovepy.read(str(vrt_path))

# Mask nodata values
nodata = profile.get("nodata")
if nodata is not None:
    elevation = np.where(elevation == nodata, np.nan, elevation)

# Plot elevation
# If elevation has shape (bands, rows, cols), take the first band
elev_2d = elevation[0] if elevation.ndim == 3 else elevation

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(elev_2d, cmap="terrain")
cbar = fig.colorbar(im, ax=ax, label="Elevation (ft)", shrink=0.8)
ax.set_title("Frankfort, KY — DEM Phase 3 Elevation")
ax.set_xlabel("Column")
ax.set_ylabel("Row")
plt.tight_layout()
plt.show()

print(f"Elevation range: {np.nanmin(elev_2d):.1f} - {np.nanmax(elev_2d):.1f} ft")

## Compute Hillshade

A hillshade simulates illumination on the terrain surface, making it easier
to interpret topography. We'll compute it from the DEM using numpy gradients.

The standard illumination parameters are:
- Azimuth: 315 degrees (northwest)
- Altitude: 45 degrees above horizon

In [ ]:
def compute_hillshade(dem, azimuth=315, altitude=45, cell_size=1.0):
    """Compute hillshade from a DEM array using numpy gradient.

    Parameters
    ----------
    dem : numpy.ndarray
        2D elevation array.
    azimuth : float
        Sun azimuth in degrees (0=north, clockwise).
    altitude : float
        Sun altitude in degrees above the horizon.
    cell_size : float
        Cell size for gradient computation.

    Returns
    -------
    numpy.ndarray
        Hillshade array with values 0-255.
    """
    # Convert angles to radians
    azimuth_rad = np.radians(360 - azimuth + 90)
    altitude_rad = np.radians(altitude)

    # Compute gradient (slope in x and y directions)
    dy, dx = np.gradient(dem, cell_size)

    # Compute slope and aspect
    slope = np.arctan(np.sqrt(dx**2 + dy**2))
    aspect = np.arctan2(-dy, dx)

    # Compute hillshade
    hillshade = (
        np.sin(altitude_rad) * np.cos(slope)
        + np.cos(altitude_rad) * np.sin(slope) * np.cos(azimuth_rad - aspect)
    )

    # Scale to 0-255
    hillshade = np.clip(hillshade * 255, 0, 255).astype(np.uint8)
    return hillshade


# Compute hillshade from the DEM
# Use pixel resolution from the profile for accurate gradient computation
cell_size = abs(profile["transform"][0]) if "transform" in profile else 1.0
hillshade = compute_hillshade(elev_2d, cell_size=cell_size)

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(hillshade, cmap="gray")
ax.set_title("Frankfort, KY — Hillshade (azimuth=315, altitude=45)")
ax.set_xlabel("Column")
ax.set_ylabel("Row")
plt.tight_layout()
plt.show()

## Overlay Hillshade + Elevation

Combine the hillshade and elevation into a single composite visualization.
The hillshade provides shading while the elevation provides color.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel 1: Elevation only
im0 = axes[0].imshow(elev_2d, cmap="terrain")
axes[0].set_title("Elevation")
fig.colorbar(im0, ax=axes[0], label="Elevation (ft)", shrink=0.6)

# Panel 2: Hillshade only
axes[1].imshow(hillshade, cmap="gray")
axes[1].set_title("Hillshade")

# Panel 3: Composite — elevation color draped over hillshade
# Normalize elevation to 0-1 for blending
elev_norm = (elev_2d - np.nanmin(elev_2d)) / (np.nanmax(elev_2d) - np.nanmin(elev_2d))
hs_norm = hillshade.astype(float) / 255.0

# Create the composite: hillshade modulates the elevation colormap
axes[2].imshow(hillshade, cmap="gray", alpha=1.0)
im2 = axes[2].imshow(elev_2d, cmap="terrain", alpha=0.5)
axes[2].set_title("Elevation + Hillshade")
fig.colorbar(im2, ax=axes[2], label="Elevation (ft)", shrink=0.6)

for ax in axes:
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")

plt.suptitle("Frankfort, KY — DEM Phase 3 Analysis", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()